# kt-aivle-big-proj-vlm — Colab 서버 운영 가이드

**사전 준비**
1. 상단 메뉴 → 런타임 → 런타임 유형 변경 → **GPU (T4 이상)** 선택
2. 좌측 사이드바 🔑 아이콘 (Secrets) → `HF_TOKEN` 키로 HuggingFace 토큰 저장

셀을 위에서 아래로 순서대로 실행하세요.

| 엔드포인트 | 설명 |
|---|---|
| `GET  /health` | 헬스체크 |
| `POST /vlm/reports/daily` | 일일 보고서 생성 |
| `GET  /docs` | Swagger UI |

## 1단계: GPU 확인

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU를 찾을 수 없습니다. 런타임 유형을 GPU로 변경해 주세요.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

## 2단계: 레포 클론

In [ ]:
import os

REPO_DIR = "/content/kt-aivle-big-proj-vlm"

if os.path.exists(REPO_DIR):
    print("이미 클론됨 — 최신화 중...")
    !git -C {REPO_DIR} pull
else:
    !git clone -b develop https://github.com/aivle-bigproject-16/kt-aivle-big-proj-vlm.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"작업 디렉터리: {os.getcwd()}")

## 3단계: 의존성 설치

> `transformers`를 git 최신 버전으로 설치하므로 3~5분 소요될 수 있습니다.

In [ ]:
!pip install -q -r requirements.txt
print("✅ 설치 완료")

## 4단계: HuggingFace 토큰 설정

좌측 🔑 Secrets 탭에 `HF_TOKEN`이 등록되어 있어야 합니다.

In [ ]:
import os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN 시크릿이 설정되지 않았습니다. 좌측 🔑 탭을 확인하세요.")

os.environ["HF_TOKEN"] = hf_token
print("✅ HF_TOKEN 설정 완료")

## 5단계: cloudflared 설치

Colab은 외부에서 직접 접근할 수 없으므로 Cloudflare Quick Tunnel로 공개 URL을 생성합니다.  
(무료, 회원가입 불필요)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
!/content/cloudflared --version

## 6단계: FastAPI 서버 시작 (포트 8080)

uvicorn을 백그라운드에서 실행합니다. 모델 로드까지 **5~10분** 소요됩니다.

In [ ]:
import subprocess, time, requests, os
import psutil, torch

REPO_DIR = "/content/kt-aivle-big-proj-vlm"

if not os.path.exists(REPO_DIR):
    raise RuntimeError("레포 디렉터리가 없습니다. 2단계(클론) 셀을 먼저 실행하세요.")

# 8080 포트 선점 프로세스 제거 (재실행 시 충돌 방지)
subprocess.run("fuser -k 8080/tcp 2>/dev/null || true", shell=True)
time.sleep(1)

def mem_status():
    ram = psutil.virtual_memory()
    ram_used = ram.used / 1e9
    ram_total = ram.total / 1e9
    vram_used = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    return f"RAM {ram_used:.1f}/{ram_total:.1f}GB  |  VRAM {vram_used:.1f}/{vram_total:.1f}GB"

server_log = open("/content/server.log", "w")
server_proc = subprocess.Popen(
    [
        "python", "-m", "uvicorn", "app.main:app",
        "--host", "0.0.0.0",
        "--port", "8080",
        "--log-level", "info",
    ],
    cwd=REPO_DIR,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)
print(f"서버 PID: {server_proc.pid}")
print("모델 로드 대기 중...\n")
print(f"{'경과':>6}  {'상태':<10}  메모리")
print("-" * 50)

for i in range(120):  # 최대 10분
    time.sleep(5)
    elapsed = f"{(i+1)*5}초"
    try:
        r = requests.get("http://localhost:8080/health", timeout=2)
        if r.status_code == 200:
            print(f"{elapsed:>6}  {'✅ 준비완료':<10}  {mem_status()}")
            break
    except Exception:
        print(f"{elapsed:>6}  {'대기중':<10}  {mem_status()}")
else:
    print("\n⚠️  타임아웃 — 서버 로그를 확인하세요")
    !tail -30 /content/server.log

## 7단계: 외부 공개 URL 생성 (cloudflared 터널)

In [ ]:
import subprocess, threading, re, time

cf_log = open("/content/cf.log", "w")
cf_proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8080"],
    stdout=cf_log,
    stderr=subprocess.PIPE,
    text=True,
)

public_url = None

def _find_url():
    global public_url
    for line in cf_proc.stderr:
        cf_log.write(line)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            public_url = m.group()
            break

t = threading.Thread(target=_find_url, daemon=True)
t.start()
t.join(timeout=30)

if public_url:
    print(f"🌐 공개 URL: {public_url}")
    print(f"   헬스체크:     {public_url}/health")
    print(f"   API 문서:     {public_url}/docs")
    print(f"   보고서 생성:  POST {public_url}/vlm/reports/daily")
else:
    print("⚠️  URL을 찾지 못했습니다. /content/cf.log를 확인하세요.")
    !cat /content/cf.log

---
## (선택) API 테스트

서버가 준비된 후 아래 셀로 보고서 생성을 테스트할 수 있습니다.

In [ ]:
import requests, json
from IPython.display import display, Markdown

BASE_URL = "http://localhost:8080"  # 외부에서 호출 시 public_url 로 변경

payload = {
    "daily_data": {
        "reportDate": "2026-07-28",
        "summaryData": {
            "totalCount": 200,
            "passCount": 185,
            "rejectCount": 12,
            "failedCount": 3,
            "prevTotalCount": 180,
            "prevRejectCount": 10,
            "defects": [
                {"defectType": "CRACK",   "count": 7},
                {"defectType": "SCRATCH", "count": 5},
            ],
        },
    }
}

resp = requests.post(f"{BASE_URL}/vlm/reports/daily", json=payload, timeout=600)
resp.raise_for_status()

data = resp.json()
print(f"상태: {data['status']}")
print(f"제목: {data['title']}")
display(Markdown(data["content"] or data["failureReason"]))

## (선택) 서버 종료

In [ ]:
server_proc.terminate()
cf_proc.terminate()
print("서버 및 터널 종료됨")